# 🌱 Dry Beans Classification using Spark ML

## 📌 Overview

In this project, we build a **machine learning classification pipeline** using **Apache Spark** to classify dry bean types based on their physical and geometrical properties.

This project simulates a real-world scenario where data engineers and machine learning engineers collaborate to process structured data, build scalable classification models, and generate predictions.

The dataset contains multiple bean classes, making this a **multiclass classification problem**, which is common in real-world machine learning applications.

---

## 🎯 Objectives

- Perform **ETL operations** on raw dataset  
- Inspect and preprocess structured data  
- Convert categorical labels into numerical format  
- Build a **feature engineering pipeline** using Spark ML  
- Train a **Logistic Regression classification model**  
- Evaluate model performance using multiple classification metrics  
- Perform inference on new unseen data  

---

## 🛠️ Technologies Used

- **Apache Spark (PySpark)**
- **Spark ML (Machine Learning Library)**
- **Python**

---

## 📊 Dataset

The dataset contains over **13,612 observations** of dry beans, each described by geometric and shape-related features.

Main features include:

- `Area` — size of the bean  
- `Perimeter` — boundary length  
- `MajorAxisLength` — longest axis of the bean  
- `MinorAxisLength` — shortest axis  
- `AspectRatio` — ratio between axes  
- `Eccentricity` — shape elongation  
- `Compactness`, `Solidity`, `Roundness` — shape descriptors  

Target variable:

- `Class` — bean type (to be predicted)

💡 This is a **multiclass classification problem**, where the model learns to distinguish between different bean varieties.

---

## 💡 Project Structure

The pipeline consists of the following stages:

1. Data Loading  
2. Data Cleaning and Transformation  
3. Label Encoding  
4. Feature Engineering  
5. Machine Learning Pipeline  
6. Model Evaluation  
7. Inference on New Data  

---

💡 This project demonstrates how **data engineering and machine learning pipelines integrate together** in a scalable Spark environment.

## 1. Import + Spark Initialization

In this section, we initialize the Spark environment and import the required libraries for building a Spark ML classification pipeline.

We use:
- **SparkSession** to work with Spark DataFrames
- **StringIndexer** to convert categorical labels into numerical format
- **VectorAssembler** to combine input features into a single vector
- **LogisticRegression** for multiclass classification
- **MulticlassClassificationEvaluator** for model evaluation

In [3]:
# Initialize Spark environment

import findspark
findspark.init()

from pyspark.sql import SparkSession

from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Create Spark Session

spark = SparkSession \
    .builder \
    .appName("Dry Beans Classification Pipeline") \
    .getOrCreate()

sc = spark.sparkContext

print("Spark session initialized successfully")
print(f"Spark version: {spark.version}")

Spark session initialized successfully
Spark version: 3.5.1


## 2. Data Loading

In this section, we load the raw dry beans dataset into a Spark DataFrame.

The dataset is stored in CSV format and contains shape-based features used to classify different bean types.

In [4]:
# Load raw dataset

beans_df = spark.read.csv(
    "data/drybeans.csv",
    header=True,
    inferSchema=True
)

# Preview data

beans_df.show(5)

+-----+---------+---------------+---------------+------------+------------+----------+-------------+-----------+-----------+-----------+-----------+------------+------------+------------+------------+-----+
| Area|Perimeter|MajorAxisLength|MinorAxisLength|AspectRation|Eccentricity|ConvexArea|EquivDiameter|     Extent|   Solidity|  roundness|Compactness|ShapeFactor1|ShapeFactor2|ShapeFactor3|ShapeFactor4|Class|
+-----+---------+---------------+---------------+------------+------------+----------+-------------+-----------+-----------+-----------+-----------+------------+------------+------------+------------+-----+
|28395|  610.291|    208.1781167|     173.888747| 1.197191424| 0.549812187|     28715|  190.1410973|0.763922518|0.988855999|0.958027126|0.913357755| 0.007331506| 0.003147289| 0.834222388| 0.998723889|SEKER|
|28734|  638.018|    200.5247957|    182.7344194| 1.097356461| 0.411785251|     29172|  191.2727505|0.783968133|0.984985603|0.887033637|0.953860842| 0.006978659| 0.00356362

In [5]:
# Count total number of rows

row_count_raw = beans_df.count()

print("Total rows in raw dataset:", row_count_raw)

Total rows in raw dataset: 13611


The dataset was successfully loaded into a Spark DataFrame.

It contains over 13,000 records and multiple numerical features describing bean characteristics.

## 3. Schema Inspection + Data Understanding

Before building the model, we inspect the dataset structure and understand the data types and distribution.

This helps us:
- Verify that features are correctly typed  
- Identify the target variable  
- Understand class distribution for classification  

In [6]:
# Inspect schema

beans_df.printSchema()

root
 |-- Area: integer (nullable = true)
 |-- Perimeter: double (nullable = true)
 |-- MajorAxisLength: double (nullable = true)
 |-- MinorAxisLength: double (nullable = true)
 |-- AspectRation: double (nullable = true)
 |-- Eccentricity: double (nullable = true)
 |-- ConvexArea: integer (nullable = true)
 |-- EquivDiameter: double (nullable = true)
 |-- Extent: double (nullable = true)
 |-- Solidity: double (nullable = true)
 |-- roundness: double (nullable = true)
 |-- Compactness: double (nullable = true)
 |-- ShapeFactor1: double (nullable = true)
 |-- ShapeFactor2: double (nullable = true)
 |-- ShapeFactor3: double (nullable = true)
 |-- ShapeFactor4: double (nullable = true)
 |-- Class: string (nullable = true)



In [7]:
# Preview selected columns

beans_df.select("Area", "Perimeter", "Solidity", "roundness", "Compactness", "Class").show(5)

+-----+---------+-----------+-----------+-----------+-----+
| Area|Perimeter|   Solidity|  roundness|Compactness|Class|
+-----+---------+-----------+-----------+-----------+-----+
|28395|  610.291|0.988855999|0.958027126|0.913357755|SEKER|
|28734|  638.018|0.984985603|0.887033637|0.953860842|SEKER|
|29380|   624.11|0.989558774|0.947849473|0.908774239|SEKER|
|30008|  645.884|0.976695743|0.903936374|0.928328835|SEKER|
|30140|  620.134| 0.99089325|0.984877069|0.970515523|SEKER|
+-----+---------+-----------+-----------+-----------+-----+
only showing top 5 rows



In [8]:
# Check class distribution

beans_df.groupBy("Class").count().orderBy("count", ascending=False).show()

+--------+-----+
|   Class|count|
+--------+-----+
|DERMASON| 3546|
|    SIRA| 2636|
|   SEKER| 2027|
|   HOROZ| 1928|
|    CALI| 1630|
|BARBUNYA| 1322|
|  BOMBAY|  522|
+--------+-----+



### 🔍 Initial Observations

- All feature columns are **numerical (integer and double types)** and suitable for machine learning  
- The target variable is **Class**, which contains multiple bean categories (e.g., DERMASON, SIRA, SEKER)  
- The dataset represents a **multiclass classification problem** with 7 distinct classes  
- The class distribution is **uneven** (e.g., DERMASON ≈ 3500 vs BOMBAY ≈ 500), which may impact model performance and bias predictions toward dominant classes  

💡 Next step: convert the categorical target column into numerical labels for model training.

## 4. Label Encoding

Machine learning models require numerical input, so we convert the categorical target column **Class** into a numerical column named **label**.

We use **StringIndexer**, which assigns a unique numeric index to each category.

This step is essential for training classification models in Spark ML.

In [9]:
# Convert categorical label to numeric

indexer = StringIndexer(
    inputCol="Class",
    outputCol="label"
)

beans_indexed = indexer.fit(beans_df).transform(beans_df)

# Preview results

beans_indexed.select("Class", "label").show(10)

+-----+-----+
|Class|label|
+-----+-----+
|SEKER|  2.0|
|SEKER|  2.0|
|SEKER|  2.0|
|SEKER|  2.0|
|SEKER|  2.0|
|SEKER|  2.0|
|SEKER|  2.0|
|SEKER|  2.0|
|SEKER|  2.0|
|SEKER|  2.0|
+-----+-----+
only showing top 10 rows



This confirms that each categorical class has been successfully mapped to a numerical label.

In [10]:
# Check label distribution

beans_indexed.groupBy("label").count().orderBy("count", ascending=False).show()

+-----+-----+
|label|count|
+-----+-----+
|  0.0| 3546|
|  1.0| 2636|
|  2.0| 2027|
|  3.0| 1928|
|  4.0| 1630|
|  5.0| 1322|
|  6.0|  522|
+-----+-----+



### 🔍 Label Encoding Results

- Each bean class is now represented as a numerical label  
- The transformation enables compatibility with Spark ML algorithms  
- Class imbalance remains present and should be considered during model evaluation  

💡 Next step: assemble feature columns into a single feature vector.

## 5. Feature Engineering

In this section, we prepare the input features for machine learning.

Spark ML models require all input columns to be combined into a single vector column named **features**.

We use **VectorAssembler** to combine selected numerical columns into one feature vector.

In [11]:
# Define feature columns

feature_columns = [
    "Area",
    "Perimeter",
    "Solidity",
    "roundness",
    "Compactness"
]

# Assemble features into a single vector column

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

beans_transformed = assembler.transform(beans_indexed)

# Preview assembled features and label

beans_transformed.select("features", "label").show(10, truncate=False)

+-----------------------------------------------------+-----+
|features                                             |label|
+-----------------------------------------------------+-----+
|[28395.0,610.291,0.988855999,0.958027126,0.913357755]|2.0  |
|[28734.0,638.018,0.984985603,0.887033637,0.953860842]|2.0  |
|[29380.0,624.11,0.989558774,0.947849473,0.908774239] |2.0  |
|[30008.0,645.884,0.976695743,0.903936374,0.928328835]|2.0  |
|[30140.0,620.134,0.99089325,0.984877069,0.970515523] |2.0  |
|[30279.0,634.927,0.989509804,0.943851783,0.923725952]|2.0  |
|[30477.0,670.033,0.984081369,0.853079869,0.933373552]|2.0  |
|[30519.0,629.727,0.989366875,0.967109244,0.925480392]|2.0  |
|[30685.0,635.681,0.988435769,0.954239808,0.925658498]|2.0  |
|[30834.0,631.934,0.990809769,0.97027823,0.91212543]  |2.0  |
+-----------------------------------------------------+-----+
only showing top 10 rows



### 🔍 Feature Engineering Results

- Selected numerical columns (**Area, Perimeter, Solidity, roundness, Compactness**) were successfully combined into a single **features** vector  
- Each row now contains a dense numerical vector representing the physical characteristics of a bean  
- The target column **label** is correctly aligned with the feature vector and ready for model training  
- The preview shows consistent vector structure and no missing values in the selected features  

💡 Next step: split the dataset into training and testing sets for model training and evaluation.

## 6. Train/Test Split

To evaluate the model properly, we split the dataset into **training** and **testing** subsets.

- **70%** of the data is used for training the model  
- **30%** is used for evaluating model performance  

We set a fixed **seed (42)** to ensure reproducibility of results.

In [12]:
# Split dataset into training and testing sets

training_data, testing_data = beans_transformed.randomSplit(
    [0.7, 0.3],
    seed=42
)

print("Training data count:", training_data.count())
print("Testing data count:", testing_data.count())

Training data count: 9634
Testing data count: 3977


### 🔍 Data Split Results

- The dataset was successfully divided into training and testing subsets  
- The training set will be used to fit the model  
- The testing set will be used to evaluate model performance on unseen data  
- The split ensures that the model generalizes beyond the training data  

💡 Next step: build and train a Logistic Regression classification model.

## 7. Model Training (Logistic Regression)

In this step, we build and train a **Logistic Regression model** for multiclass classification.

Logistic Regression is a widely used algorithm for classification tasks and works well with numerical feature vectors.

The model will learn relationships between input features and the target label to classify different bean types.

In [13]:
# Initialize Logistic Regression model

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label"
)

# Train the model

model = lr.fit(training_data)

print("Model trained successfully")

Model trained successfully


### 🔍 Model Training Results

- The model has been fitted on the training dataset
- Logistic Regression is now ready to generate predictions on unseen data
- Training was performed on the training subset only

💡 Next step: evaluate the model using multiple classification metrics.

## 8. Model Evaluation

In this section, we evaluate the performance of the trained classification model using multiple metrics.

We use:
- **Accuracy** — overall correctness of predictions  
- **Precision** — correctness of positive predictions  
- **Recall** — ability to find all relevant instances  
- **F1 Score** — balance between precision and recall  

These metrics provide a comprehensive understanding of model performance.

In [14]:
# Make predictions on testing data

predictions = model.transform(testing_data)

# Preview predictions

predictions.select("features", "label", "prediction", "probability").show(10, truncate=False)

+-----------------------------------------------------+-----+----------+---------------------------------------------------------------------------------------------------------------------------------------------------------+
|features                                             |label|prediction|probability                                                                                                                                              |
+-----------------------------------------------------+-----+----------+---------------------------------------------------------------------------------------------------------------------------------------------------------+
|[20548.0,524.736,0.986698679,0.937772948,0.87923312] |0.0  |0.0       |[0.9999780914529544,1.39505825822365E-5,7.957959773725834E-6,2.6770056557055316E-12,2.977284198417573E-17,2.01278011489675E-12,3.3783643966313415E-39]   |
|[21101.0,533.701,0.983179573,0.930929662,0.88417697] |0.0  |0.0       |[0.9999579236852413,

In [15]:
# Evaluate model performance

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
accuracy = evaluator.evaluate(predictions)

evaluator.setMetricName("weightedPrecision")
precision = evaluator.evaluate(predictions)

evaluator.setMetricName("weightedRecall")
recall = evaluator.evaluate(predictions)

evaluator.setMetricName("f1")
f1_score = evaluator.evaluate(predictions)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1_score:.4f}")

Accuracy: 0.9140
Precision: 0.9145
Recall: 0.9140
F1 Score: 0.9141


### 🔍 Model Evaluation Results

-  The model achieved solid performance across multiple evaluation metrics  
- **Accuracy: 0.9140** indicates correct classification for the majority of samples  
- **Precision: 0.9145** shows that predicted classes are highly reliable  
- **Recall: 0.9140** confirms the model effectively captures relevant instances  
- **F1 Score: 0.9141** demonstrates a well-balanced performance between precision and recall  

💡 The model demonstrates consistent performance across all metrics, indicating good generalization on unseen data.

##  9. Model Persistence & Export

In this step, we export prediction results and demonstrate how the trained model can be saved for reuse.

For local development, predictions are saved as a CSV file.
Model persistence is included as a production-style reference.

In [16]:
# Save predictions as CSV (local development)
predictions.toPandas().to_csv("output/classification_predictions.csv", index=False)
print("Predictions saved as CSV")


Predictions saved as CSV


In [2]:
# Save model (production-style reference)
# model.write().overwrite().save("output/classification_model")

# Load model example
# from pyspark.ml.classification import LogisticRegressionModel
# loaded_model = LogisticRegressionModel.load("output/classification_model")

💾 The prediction results are exported as a CSV file for local development and analysis.

⚠️ Note: Model persistence in Spark may require proper Hadoop configuration.
This step is included as a production-style reference.

## 10. Inference Reference

In this step, we demonstrate how the trained model can be used to make predictions on new, unseen data.

⚠️ In some local Windows environments, Spark actions may fail due to Spark/Python/Hadoop configuration issues.  
For this reason, the inference block is provided as a production-style reference.

In [17]:
# Inference example on new unseen data
# This block can be executed in a properly configured Spark environment.

# new_data = spark.createDataFrame([
#     (30000.0, 600.0, 0.98, 0.95, 0.91)
# ], [
#     "Area",
#     "Perimeter",
#     "Solidity",
#     "roundness",
#     "Compactness"
# ])

# new_data_transformed = assembler.transform(new_data)

# prediction = model.transform(new_data_transformed)

# prediction.select("prediction", "probability").show()

### 🔍 Inference Reference

- The inference block demonstrates how the trained model can be reused for predictions on new data  
- In a properly configured Spark environment, the model can classify new bean samples without retraining  
- This step reflects real-world deployment logic for classification models  

⚠️ In some local environments, Spark actions may fail due to Spark/Python/Hadoop configuration issues.  
For this reason, this inference block is provided as a production-style reference.

## 11. Stop Spark Session

In this final step, we properly stop the Spark session to release system resources.

This is an important practice in real-world data engineering workflows.

In [19]:
# Stop Spark session

spark.stop()

print("Spark session stopped successfully")

Spark session stopped successfully


---

## 12. Final Conclusion

This project demonstrates a complete **end-to-end classification pipeline** using **Apache Spark ML**.

### 🔧 Technical Summary

- Built a scalable **machine learning pipeline** using Spark  
- Converted categorical labels into numerical format using **StringIndexer**  
- Engineered features using **VectorAssembler**  
- Trained a **Logistic Regression model** for multiclass classification  
- Evaluated model performance using multiple metrics (Accuracy, Precision, Recall, F1)  

---

### 📊 Model Performance

- **Accuracy: 0.9140**  
- **Precision: 0.9145**  
- **Recall: 0.9140**  
- **F1 Score: 0.9141**

The model achieved solid and balanced performance across all evaluation metrics, indicating reliable classification results.

---

### 💡 Business & Practical Value

- The model can automatically classify different types of dry beans based on physical characteristics  
- This approach can be applied in **agriculture, quality control, and food processing industries**  
- The pipeline supports scalable processing of large datasets using Spark  

---

### 🚀 Key Takeaways

- Spark ML enables efficient handling of **large-scale classification problems**  
- Proper feature engineering and label encoding are critical for model performance  
- The pipeline design allows easy extension to production environments  

---

💡 This project highlights how **data engineering and machine learning pipelines integrate together** to deliver scalable and production-ready classification solutions.